# Forward advection of egg-weighted particles

Lagrangian simulation behind the larval dispersal analysis: passive particles
representing toothfish eggs and larvae are released over the spawning area and
advected forward for 18 weeks.

Written by A. Nalivaev, using the LAMTA software (LAgrangian Manifolds Tracking
Algorithm, Rousselet et al. 2025). Initial version January 2025, amended May
2025 and January 2026.

**Inputs**
- `release_area.geojson`: polygon of the release area, encompassing the spawning
  hotspot.
- DUACS satellite altimetry (surface geostrophic velocities), 0.125 degree,
  daily, June to December, 2000 to 2023. Product
  SEALEVEL_GLO_PHY_L4_MY_008_047, doi:10.48670/moi-00148.

**Pipeline**
- particles are laid on a regular 0.05 degree grid inside the release area;
- one release per week, on the Thursday of weeks 23 to 31 of each year. Those are
  the spawning weeks 22 to 30, shifted by the one week the eggs take to rise from
  about 1500 m to the surface mixed layer (Parker et al. 2021); no horizontal
  drift is applied during that ascent;
- each release is advected forward for 18 weeks (the pelagic larval duration).

**Output**
- one CSV per release week and year,
  `advection_18weeks_from_week<WW>_<YYYY>.csv`, holding the daily longitude and
  latitude of every particle.

Seabed depth along the trajectories, the beaching rule and the egg weighting are
applied downstream, in `larval_dispersal/01_load_trajectories.R`.

## Configuration

The only cell to edit: where the inputs are, and where the output goes.

In [ ]:
from pathlib import Path

# Root of the data archive (see the repository README).
DATA_DIR = Path("data")

# DUACS velocity fields, one subdirectory per year.
DUACS_DIR = DATA_DIR / "DUACS_0125"

# Where the advection output is written.
OUT_DIR = Path("outputs") / "advection_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2000, 2024)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import geopandas as gpd

import datetime as dt
from datetime import date, timedelta

from lamta.Diagnostics_mod import ParticleSet, Lagrangian
from lamta.Load_nc_mod import loadCMEMSuv

## Step 1: initialize the particles

Read the release-area polygon, lay a regular 0.05 degree grid over its bounding
box, and keep the grid points that fall inside it.

In [ ]:
release_area = gpd.read_file(DATA_DIR / "release_area.geojson")

df_coordinates = release_area['geometry'].get_coordinates()

delta = 0.05
lon_grid = np.linspace(np.min(df_coordinates['x']), np.max(df_coordinates['x']),
                       int(1 / delta * (np.max(df_coordinates['x']) - np.min(df_coordinates['x']))))
lat_grid = np.linspace(np.min(df_coordinates['y']), np.max(df_coordinates['y']),
                       int(1 / delta * (np.max(df_coordinates['y']) - np.min(df_coordinates['y']))))

X_grid, Y_grid = np.meshgrid(lon_grid, lat_grid)
X_grid_line = np.reshape(X_grid, np.shape(X_grid)[0] * np.shape(X_grid)[1])
Y_grid_line = np.reshape(Y_grid, np.shape(Y_grid)[0] * np.shape(Y_grid)[1])

df = pd.DataFrame(data={'lon': pd.Series(X_grid_line), 'lat': pd.Series(Y_grid_line)})
points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat))

# Keep the grid points that fall within the release polygon.
inside = [points.geometry[i].within(release_area['geometry'])[0]
          for i in range(points.geometry.count())]
within_points = points[inside]

print("{} particles per release".format(len(within_points)))
within_points.to_csv(OUT_DIR / "initial_positions_005degree_spacing.csv")

## Step 2: release dates

For a given year, find the Thursday (the middle of the week) of each release
week, and pair it with the end of the 18-week run. LAMTA takes dates as
ordinals.

The week numbers come from the MNHN stock model, which places spawning in weeks
22 to 30 from female abundance; the release weeks are those plus one.

In [ ]:
FIRST_WEEK = 23        # spawning week 22 plus one week of egg ascent
LAST_WEEK = 31         # spawning week 30 plus one week
WEEKS_SIMU = 18        # pelagic larval duration
NUMDAYS = WEEKS_SIMU * 7


def release_dates(year, first_week=FIRST_WEEK, last_week=LAST_WEEK, numdays=NUMDAYS):
    """Start and end ordinals of each weekly release of a given year."""
    weeks, spans = [], []
    for week in range(first_week, last_week + 1):
        d = "{}-W{}".format(year, week)
        thursday = dt.datetime.strptime(d + '-4', "%Y-W%W-%w")
        day1 = dt.datetime.strptime(thursday.strftime('%Y-%m-%d'), '%Y-%m-%d').date()
        day1j = dt.datetime.toordinal(day1)
        weeks.append(week)
        spans.append(np.array([day1j, day1j + numdays]))
    return weeks, spans

## Step 3: advection

For each year, load the daily geostrophic velocity fields once, then advect every
weekly release forward for 18 weeks with a fourth-order Runge-Kutta scheme at a
6-hour timestep.

The scheme produces four positions a day; only the whole dates are kept, giving
one longitude and one latitude column per day. Coordinates are rounded to two
decimals (about 1 km).

This is the long part: a full year of daily altimetry is loaded, then nine
18-week advections are run, for each of 24 years.

In [ ]:
varn = {'longitude': 'longitude', 'latitude': 'latitude',
        'u': 'ugos', 'v': 'vgos', 'ssh': 'ssha'}

px = np.array(within_points['lon'])
py = np.array(within_points['lat'])
numstep = 4 * NUMDAYS

for year in YEARS:

    # Load the whole season at once: June 1st to January 1st of the next year, a
    # wider window than the runs need, but read only once per year.
    sdate = date(year, 6, 1)
    edate = date(year + 1, 1, 1)
    all_days = [d.strftime('%Y%m%d')
                for d in pd.date_range(sdate, edate - timedelta(days=1), freq='d')]

    field = loadCMEMSuv(all_days, str(DUACS_DIR / str(year)) + '/', varn, unit='deg/d')

    weeks, spans = release_dates(year)

    for week, pt in zip(weeks, spans):

        pset = ParticleSet.from_input(pt, px, py, fieldset=field, mode='forward')
        trjf = pset.rk4flat(Lagrangian.interpf, numstep, coordinates='spherical')

        day_start = dt.datetime.fromordinal(round(np.array(trjf['trjt'])[0])).strftime('%Y%m%d')
        df_output = pd.DataFrame(data={
            '{} Longitude'.format(day_start): [round(v, 2) for v in np.array(trjf['trjx'])[0]],
            '{} Latitude'.format(day_start): [round(v, 2) for v in np.array(trjf['trjy'])[0]]})

        ncol = 2
        for i in np.arange(2, len(trjf['trjt']) + 1):
            timestep = np.array(trjf['trjt'])[i] if i < len(trjf['trjt']) else pt[1]

            # One position every 6 hours; keep the whole dates only.
            if timestep % 1 == 0:
                day = dt.datetime.fromordinal(round(timestep)).strftime('%Y%m%d')
                df_output.insert(ncol, '{} Longitude'.format(day),
                                 [round(v, 2) for v in np.array(trjf['trjx'])[i - 1]], True)
                ncol += 1
                df_output.insert(ncol, '{} Latitude'.format(day),
                                 [round(v, 2) for v in np.array(trjf['trjy'])[i - 1]], True)
                ncol += 1

        out = OUT_DIR / 'advection_{}weeks_from_week{}_{}.csv'.format(WEEKS_SIMU, week, year)
        df_output.to_csv(out)
        print("written", out.name)